# TrustRAG C1 - ModernBERT span detector

Thin runner. All logic lives in `src/c1_detector/`; this notebook clones the repo,
installs what Kaggle is missing, points the code at the data and calls it. Anything
worth debugging should be debugged locally against `configs/c1_smoke.yaml`, not here.

## Before running - do these by hand, in this order

1. **Upload the data.** `data/processed/` is gitignored, so it does not arrive with
   the clone. On your machine, upload both files as a **private Kaggle Dataset**:
   `data/processed/ragtruth_train.jsonl` and `ragtruth_test.jsonl` (about 64 MB together).
   Name it `ragtruth-processed`. Add it to this notebook: **File -> Add Data -> Your Datasets**.
   Then confirm the path in the DATA_DIR cell - Kaggle mounts it at
   `/kaggle/input/<dataset-slug>/`.
2. **Add secrets.** Add-ons -> Secrets:
   - `GITHUB_TOKEN` - a fine-grained PAT with read access to the repo. Required while
     the repo is private.
   - `WANDB_API_KEY` - optional. Without it training runs fine and logs nothing.
   - `HF_TOKEN` - optional. ModernBERT-base is public; a token only raises rate limits.
3. **Turn on the GPU.** Settings -> Accelerator -> GPU P100 (or T4 x2).
4. **Turn on internet.** Needed for the clone, pip and the ModernBERT download.
5. **Save & Run All (Commit)**, not interactive run. It executes in the background,
   survives the browser closing, and keeps the output. This is the whole point of
   using Kaggle.

Watch the first evaluation block. If `answers truncated` is anything but 0, stop -
`max_length` is cutting labels off the end of answers and every number after that
is wrong.

In [ ]:
import os
import subprocess
import sys

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        print(f"secret {name} not set")
        return None


GITHUB_TOKEN = secret("GITHUB_TOKEN")
for name in ("WANDB_API_KEY", "HF_TOKEN"):
    value = secret(name)
    if value:
        os.environ[name] = value

REPO_DIR = "/kaggle/working/trustrag"
url = (
    f"https://{GITHUB_TOKEN}@github.com/Wimukthi316/TrustRAG.git"
    if GITHUB_TOKEN
    else "https://github.com/Wimukthi316/TrustRAG.git"
)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", url, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle ships torch already; reinstalling it wastes minutes and risks a CUDA
# mismatch. Only the pieces Kaggle is missing or has an old version of.
!pip install -q -U "transformers>=4.48" seqeval pyyaml wandb

import torch
import transformers

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"gpu {name} ({total:.1f} GB) | bf16 {torch.cuda.is_bf16_supported()}")
else:
    raise SystemExit("no GPU - set Accelerator to GPU before running")

In [ ]:
# Point the repo's relative data paths at the attached Kaggle Dataset. Symlinks
# rather than copies: 64 MB copied twice is 64 MB of session disk for nothing.
import glob
from pathlib import Path

DATA_DIR = "/kaggle/input/ragtruth-processed"  # change if your dataset slug differs

if not os.path.isdir(DATA_DIR):
    print("available inputs:", glob.glob("/kaggle/input/*"))
    raise SystemExit(f"{DATA_DIR} not found - attach the dataset and fix DATA_DIR")

processed = Path(REPO_DIR) / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)

for name in ("ragtruth_train.jsonl", "ragtruth_test.jsonl"):
    source = Path(DATA_DIR) / name
    if not source.exists():
        raise SystemExit(f"{source} missing from the attached dataset")
    target = processed / name
    if not target.exists():
        target.symlink_to(source)
    lines = sum(1 for _ in source.open(encoding="utf-8"))
    print(f"{name}: {lines:,} records")

# Expected: 15,090 train and 2,700 test, from the verified preprocessing run.
# Different numbers mean a different build_examples flag combination was used.

In [ ]:
# One quick pass to prove the data path works before spending GPU hours on it.
!python -m src.c1_detector.train_c1 \
    --config configs/c1_smoke.yaml \
    --limit 200 \
    --out-dir /kaggle/working/results/c1/kaggle-smoke \
    --run-name c1-kaggle-smoke \
    --no-wandb

In [ ]:
# The real run. Everything about it is in configs/c1_base.yaml, which is committed,
# so this result can be reconstructed later. If it OOMs, halve train.batch_size and
# double train.grad_accum in that file - the effective batch stays 8 and the run
# stays comparable to earlier ones.
!python -m src.c1_detector.train_c1 \
    --config configs/c1_base.yaml \
    --out-dir /kaggle/working/results/c1/modernbert-base

In [ ]:
# Held-out test set. These are the numbers that go in the report - the per-task
# breakdown, and the example-level F1 that compares against LettuceDetect's 79.22%.
# --dump-probs writes the per-span probabilities C2 calibrates on.
!python -m src.c1_detector.evaluate_c1 \
    --checkpoint /kaggle/working/results/c1/modernbert-base/best \
    --data data/processed/ragtruth_test.jsonl \
    --max-length 3072 \
    --batch-size 8 \
    --num-workers 2 \
    --dump-probs \
    --out-dir /kaggle/working/results/c1/test

In [ ]:
# Same evaluation over the calibration split, which the model never trained on and
# never had epochs selected on. This file is C2's input: split conformal needs
# scores from data the detector has not seen, or the coverage guarantee is void.
import json

split_ids = json.load(
    open("/kaggle/working/results/c1/modernbert-base/split_ids.json", encoding="utf-8")
)
calib_ids = set(split_ids["calib"])
print(f"calibration split: {len(calib_ids):,} responses")

calib_path = "/kaggle/working/ragtruth_calib.jsonl"
kept = 0
with open(calib_path, "w", encoding="utf-8") as out:
    for line in open("data/processed/ragtruth_train.jsonl", encoding="utf-8"):
        if json.loads(line)["id"] in calib_ids:
            out.write(line)
            kept += 1
print(f"wrote {kept:,} records to {calib_path}")
assert kept == len(calib_ids), "calibration ids did not all resolve to records"

In [ ]:
!python -m src.c1_detector.evaluate_c1 \
    --checkpoint /kaggle/working/results/c1/modernbert-base/best \
    --data /kaggle/working/ragtruth_calib.jsonl \
    --max-length 3072 \
    --batch-size 8 \
    --num-workers 2 \
    --dump-probs \
    --out-dir /kaggle/working/results/c1/calib

In [ ]:
# What to download from the committed notebook's Output tab.
#
#   results/c1/test/metrics.json           the reportable numbers
#   results/c1/test/probabilities.jsonl    C2's test-side input
#   results/c1/calib/probabilities.jsonl   C2's calibration-side input
#   results/c1/modernbert-base/summary.json + split_ids.json + history.json
#   results/c1/modernbert-base/best/       the checkpoint
#
# The clone is deleted so it does not end up in the notebook output; it is already
# on GitHub.
import shutil

for path, _, files in os.walk("/kaggle/working/results"):
    for name in sorted(files):
        full = os.path.join(path, name)
        print(f"{os.path.getsize(full)/1e6:8.2f} MB  {full}")

os.chdir("/kaggle/working")
shutil.rmtree(REPO_DIR, ignore_errors=True)